# Phase 04 — Pair Generation and Split Strategy

**Status:** Complete  
**Workflow:** Notebook-first training documentation  
**Purpose:** Design balanced training pairs and leakage-safe splits for job-fit and recommendation ranking experiments.

This notebook is the Phase 4 source of truth. It writes `reports/phase_04_pair_generation_splits.json` so baseline evaluation, model experiments, and manual validation inherit the same pair taxonomy, target distribution, split policy, required metadata, and diagnostic plan.


## Contract boundary

Phase 4 defines pair-generation and split strategy only. It does not train a model, tune score thresholds from validation labels, create manual labels, hydrate job details, or change backend/API behavior.

Model/training may generate source-grounded profile/CV-to-job pairs, weak-label component scores, pair metadata, and train/validation/test split names. Backend/API wrapper remains owner of auth, persistence, final product copy, candidate retrieval, hydrated job details, and public response shaping.


## Shared setup

### Purpose
Define deterministic paths, load Phase 2 label policy and Phase 3 normalization policy, inspect current source inventories, and create helpers used by every Phase 4 step.

### Required input
Repository root with `TODOS.md`, `reports/phase_02_label_schema_baselines.json`, `reports/phase_03_normalization_feature_design.json`, `legacy/dataset/indotech_job_cleaned.csv`, and `legacy/dataset/techtalent_profile_cleaned.csv`.

### Action
Load previous phase reports and source snapshots. Derive coarse role-family inventories from source role/title/category text for planning and diagnostics. No pair rows are generated in this planning notebook.

### Expected output
Shared constants, deterministic seed, role-family helper, source inventory, and report-writing helper.

### Verification
Setup must run with standard Python plus pandas. Generated output must be limited to `reports/phase_04_pair_generation_splits.json`.


In [7]:
from __future__ import annotations

import hashlib
import json
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "TODOS.md").exists() and (candidate / "training").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook runtime.")


ROOT = find_repo_root(Path.cwd().resolve())
REPORTS = ROOT / "reports"
PHASE2_REPORT = REPORTS / "phase_02_label_schema_baselines.json"
PHASE3_REPORT = REPORTS / "phase_03_normalization_feature_design.json"
PHASE4_REPORT = REPORTS / "phase_04_pair_generation_splits.json"
JOBS_CSV = ROOT / "legacy" / "dataset" / "indotech_job_cleaned.csv"
PROFILES_CSV = ROOT / "legacy" / "dataset" / "techtalent_profile_cleaned.csv"
SPLIT_SEED = 20260601
TARGET_PAIR_COUNT = 30000


def load_json(path: Path) -> dict[str, Any]:
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


phase2_report = load_json(PHASE2_REPORT)
phase3_report = load_json(PHASE3_REPORT)
jobs = pd.read_csv(JOBS_CSV)
profiles = pd.read_csv(PROFILES_CSV)

ROLE_FAMILY_KEYWORDS = {
    "frontend": ["frontend", "front end", "react", "vue", "angular", "web developer"],
    "backend": ["backend", "back end", "api", "server", "node", "java developer", "python developer"],
    "data_analytics": ["data analyst", "analytics", "business intelligence", "power bi", "tableau"],
    "machine_learning_ai": ["machine learning", "ml", "ai", "data scientist", "deep learning"],
    "devops_cloud": ["devops", "cloud", "sre", "site reliability", "kubernetes", "platform engineer"],
    "cybersecurity": ["security", "cyber", "devsecops", "siem", "ethical hacking"],
    "mobile": ["mobile", "android", "ios", "flutter", "react native"],
    "quality_assurance": ["qa", "quality assurance", "tester", "test engineer"],
    "product_design": ["product", "designer", "ui", "ux"],
    "hardware_iot": ["embedded", "iot", "hardware", "electronics", "microcontroller"],
    "general_software": ["software", "developer", "engineer", "programmer"],
}


def clean_text(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().lower()


def role_family_from_text(*values: Any) -> str:
    haystack = " | ".join(clean_text(value) for value in values)
    for family, keywords in ROLE_FAMILY_KEYWORDS.items():
        if any(keyword in haystack for keyword in keywords):
            return family
    return "unknown_role_family"


def stable_bucket(value: Any, modulo: int = 100) -> int:
    digest = hashlib.sha256(f"{SPLIT_SEED}:{value}".encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % modulo


job_role_families = [
    role_family_from_text(row.get("normalized_title"), row.get("title"), row.get("category"), row.get("skills_clean"))
    for _, row in jobs.iterrows()
]
profile_role_families = [
    role_family_from_text(row.get("Job_Role"), row.get("Skills"), row.get("Required_Skills"))
    for _, row in profiles.iterrows()
]

source_inventory = {
    "jobs_rows": int(len(jobs)),
    "profiles_rows": int(len(profiles)),
    "job_role_family_counts": dict(Counter(job_role_families).most_common()),
    "profile_role_family_counts": dict(Counter(profile_role_families).most_common()),
    "job_language_counts": {str(k): int(v) for k, v in jobs["language_signal"].fillna("UNKNOWN").value_counts().items()},
    "profile_experience_counts": {str(k): int(v) for k, v in profiles["Experience"].fillna("UNKNOWN").value_counts().items()},
    "job_experience_counts": {str(k): int(v) for k, v in jobs["experience_level"].fillna("UNKNOWN").value_counts().items()},
    "target_pair_count_for_planning": TARGET_PAIR_COUNT,
    "split_seed": SPLIT_SEED,
}

print(json.dumps({
    "jobs_rows": source_inventory["jobs_rows"],
    "profiles_rows": source_inventory["profiles_rows"],
    "job_role_families": source_inventory["job_role_family_counts"],
    "profile_role_families_top_5": dict(list(source_inventory["profile_role_family_counts"].items())[:5]),
    "split_seed": SPLIT_SEED,
}, indent=2))


{
  "jobs_rows": 2073,
  "profiles_rows": 69929,
  "job_role_families": {
    "general_software": 579,
    "backend": 236,
    "frontend": 227,
    "machine_learning_ai": 206,
    "data_analytics": 189,
    "quality_assurance": 145,
    "product_design": 141,
    "devops_cloud": 130,
    "cybersecurity": 109,
    "mobile": 47,
    "unknown_role_family": 45,
    "hardware_iot": 19
  },
  "profile_role_families_top_5": {
    "machine_learning_ai": 20645,
    "backend": 18599,
    "frontend": 16824,
    "devops_cloud": 5872,
    "cybersecurity": 5445
  },
  "split_seed": 20260601
}


## Step 4.1 — Pair type taxonomy

### Purpose
Define high-fit positives, medium-fit pairs, hard negatives, random negatives, same-role different-seniority pairs, and cross-role confusing pairs.

### Required input
Use Phase 2 job-fit label components, Phase 3 normalization rules, audited source identifiers, normalized skill evidence, normalized experience, language signal, and role-family evidence. Do not use existing `fit_score` as the source of truth for new pair selection.

### Action
Create a versioned taxonomy with deterministic sampling intent, label-band expectation, source evidence, exclusion rules, and minimum metadata for each pair type. Pair types must cover positive, medium, and negative cases, including confusing cases that expose ranking weaknesses.

### Expected output
A reviewable pair-type taxonomy that pair-generation code can implement without inventing categories later.

### Verification
The taxonomy must include all required pair types and all low/medium/high score bands.


In [8]:
pair_type_taxonomy = [
    {
        "pair_type": "high_fit_positive",
        "expected_band": "high",
        "target_score_range": [0.65, 1.00],
        "sampling_rule": "Same or strongly adjacent role family, high normalized skill overlap, experience meets or exceeds job requirement, requirement coverage is high, and language evidence is not UNKNOWN when enough text exists.",
        "source_evidence": ["normalized_skills", "role_family", "normalized_experience", "requirement_coverage", "language"],
        "exclusion_rules": ["Do not use current legacy fit_score as selector", "Exclude pairs with empty skills or unknown experience until reviewed"],
        "training_purpose": "Teach upper score range and strong recommendation ordering.",
    },
    {
        "pair_type": "medium_fit_pair",
        "expected_band": "medium",
        "target_score_range": [0.35, 0.64],
        "sampling_rule": "Partial skill overlap or adjacent role family with visible gaps; experience may be one band below or above but not unrelated.",
        "source_evidence": ["partial_skill_overlap", "adjacent_role_family", "bounded_experience_gap"],
        "exclusion_rules": ["Exclude pairs with no interpretable role evidence", "Do not inflate to high when only generic skills match"],
        "training_purpose": "Teach middle score range and actionable partial-match explanations.",
    },
    {
        "pair_type": "hard_negative",
        "expected_band": "low",
        "target_score_range": [0.00, 0.34],
        "sampling_rule": "Superficially similar title or shared generic skills but critical requirements, role family, or seniority do not match.",
        "source_evidence": ["generic_skill_overlap", "critical_requirement_gap", "role_or_seniority_mismatch"],
        "exclusion_rules": ["Do not sample only random unrelated pairs", "Keep a reason code for why the pair is hard"],
        "training_purpose": "Prevent over-scoring confusing low-quality matches.",
    },
    {
        "pair_type": "random_negative",
        "expected_band": "low",
        "target_score_range": [0.00, 0.25],
        "sampling_rule": "Random profile/job combinations across unrelated role families after group-safe candidate filtering.",
        "source_evidence": ["role_family_mismatch", "low_skill_overlap"],
        "exclusion_rules": ["Do not dominate the dataset", "Exclude accidental high-overlap pairs and reclassify them"],
        "training_purpose": "Provide broad low-fit background examples without making negatives too easy.",
    },
    {
        "pair_type": "same_role_different_seniority",
        "expected_band": "medium",
        "target_score_range": [0.30, 0.60],
        "sampling_rule": "Same role family and meaningful skill overlap but candidate experience is below/above required seniority enough to affect fit.",
        "source_evidence": ["same_role_family", "skill_overlap", "experience_gap"],
        "exclusion_rules": ["Do not label as high without seniority evidence", "Do not punish overqualified candidates below medium without product decision"],
        "training_purpose": "Teach seniority-sensitive scoring separately from skill matching.",
    },
    {
        "pair_type": "cross_role_confusing",
        "expected_band": "low",
        "target_score_range": [0.10, 0.40],
        "sampling_rule": "Different role families with overlapping buzzwords or tools, such as data analyst vs backend API role, DevOps vs cybersecurity, or frontend vs mobile.",
        "source_evidence": ["different_role_family", "shared_tools", "requirement_gap"],
        "exclusion_rules": ["Keep as low or low-medium unless manual review proves transferability", "Record role families explicitly"],
        "training_purpose": "Reduce false-high scores from shared tools across different work contexts.",
    },
]

required_pair_types = {
    "high_fit_positive", "medium_fit_pair", "hard_negative", "random_negative",
    "same_role_different_seniority", "cross_role_confusing",
}
covered_pair_types = {row["pair_type"] for row in pair_type_taxonomy}
covered_bands = {row["expected_band"] for row in pair_type_taxonomy}
pair_type_acceptance = {
    "required_pair_types_present": required_pair_types <= covered_pair_types,
    "bands_covered": {"low", "medium", "high"} <= covered_bands,
    "taxonomy_size": len(pair_type_taxonomy),
}

assert pair_type_acceptance["required_pair_types_present"], sorted(required_pair_types - covered_pair_types)
assert pair_type_acceptance["bands_covered"], sorted({"low", "medium", "high"} - covered_bands)
print(json.dumps(pair_type_acceptance, indent=2))


{
  "required_pair_types_present": true,
  "bands_covered": true,
  "taxonomy_size": 6
}


## Step 4.2 — Target distribution

### Purpose
Document target score distribution across 0-1 and minimum high-fit coverage needed for meaningful training.

### Required input
Use Phase 2 score-band policy, pair-type taxonomy, current gap that legacy pairs have no high-fit labels above `0.6`, and the planned dataset size. Existing weak-label distribution is evidence of the gap, not the desired future distribution.

### Action
Define target score-bin shares, pair-type shares, minimum high-fit coverage, and release gates. The distribution must keep enough high-fit positives in every split while preserving medium and negative cases.

### Expected output
A target distribution plan with per-bin and per-pair-type counts for the planning dataset size.

### Verification
Target shares must sum to 1.0. High-fit coverage must be at least 20% overall and present in validation/test splits.


In [9]:
target_score_distribution = [
    {"score_min": 0.00, "score_max": 0.20, "target_share": 0.15, "purpose": "clear random negatives and severe mismatches"},
    {"score_min": 0.21, "score_max": 0.34, "target_share": 0.15, "purpose": "hard/cross-role negatives near low-band boundary"},
    {"score_min": 0.35, "score_max": 0.49, "target_share": 0.15, "purpose": "lower-medium partial matches"},
    {"score_min": 0.50, "score_max": 0.64, "target_share": 0.15, "purpose": "upper-medium adjacent or incomplete matches"},
    {"score_min": 0.65, "score_max": 0.79, "target_share": 0.20, "purpose": "high-fit positives with some gaps"},
    {"score_min": 0.80, "score_max": 1.00, "target_share": 0.20, "purpose": "strong positives for full-range calibration"},
]

pair_type_target_distribution = [
    {"pair_type": "high_fit_positive", "target_share": 0.25, "minimum_per_split_share": 0.15},
    {"pair_type": "medium_fit_pair", "target_share": 0.20, "minimum_per_split_share": 0.15},
    {"pair_type": "hard_negative", "target_share": 0.20, "minimum_per_split_share": 0.15},
    {"pair_type": "random_negative", "target_share": 0.10, "minimum_per_split_share": 0.05},
    {"pair_type": "same_role_different_seniority", "target_share": 0.15, "minimum_per_split_share": 0.10},
    {"pair_type": "cross_role_confusing", "target_share": 0.10, "minimum_per_split_share": 0.05},
]

minimum_high_fit_coverage = {
    "overall_min_share": 0.20,
    "overall_preferred_share": 0.40,
    "minimum_validation_examples": 300,
    "minimum_test_examples": 300,
    "minimum_manual_review_candidates_per_split": 30,
    "release_gate": "Do not train a full-range scorer if validation or test split has too few high-fit examples to evaluate the 65-100 score band.",
}

score_distribution_counts = [
    {**row, "target_count": int(round(row["target_share"] * TARGET_PAIR_COUNT))}
    for row in target_score_distribution
]
pair_type_distribution_counts = [
    {**row, "target_count": int(round(row["target_share"] * TARGET_PAIR_COUNT))}
    for row in pair_type_target_distribution
]

distribution_acceptance = {
    "score_share_sum": round(sum(row["target_share"] for row in target_score_distribution), 6),
    "pair_type_share_sum": round(sum(row["target_share"] for row in pair_type_target_distribution), 6),
    "high_score_share": round(sum(row["target_share"] for row in target_score_distribution if row["score_min"] >= 0.65), 6),
    "minimum_high_fit_coverage_met": sum(row["target_share"] for row in target_score_distribution if row["score_min"] >= 0.65) >= minimum_high_fit_coverage["overall_min_share"],
}

assert distribution_acceptance["score_share_sum"] == 1.0, distribution_acceptance
assert distribution_acceptance["pair_type_share_sum"] == 1.0, distribution_acceptance
assert distribution_acceptance["minimum_high_fit_coverage_met"], distribution_acceptance
print(json.dumps({
    "score_distribution_counts": score_distribution_counts,
    "pair_type_distribution_counts": pair_type_distribution_counts,
    "acceptance": distribution_acceptance,
}, indent=2))


{
  "score_distribution_counts": [
    {
      "score_min": 0.0,
      "score_max": 0.2,
      "target_share": 0.15,
      "purpose": "clear random negatives and severe mismatches",
      "target_count": 4500
    },
    {
      "score_min": 0.21,
      "score_max": 0.34,
      "target_share": 0.15,
      "purpose": "hard/cross-role negatives near low-band boundary",
      "target_count": 4500
    },
    {
      "score_min": 0.35,
      "score_max": 0.49,
      "target_share": 0.15,
      "purpose": "lower-medium partial matches",
      "target_count": 4500
    },
    {
      "score_min": 0.5,
      "score_max": 0.64,
      "target_share": 0.15,
      "purpose": "upper-medium adjacent or incomplete matches",
      "target_count": 4500
    },
    {
      "score_min": 0.65,
      "score_max": 0.79,
      "target_share": 0.2,
      "purpose": "high-fit positives with some gaps",
      "target_count": 6000
    },
    {
      "score_min": 0.8,
      "score_max": 1.0,
      "target_share": 0.

## Step 4.3 — Group isolation

### Purpose
Define split isolation by profile ID and optional job family to reduce leakage between training and validation.

### Required input
Use audited profile/job identifiers, Phase 1 leakage controls, Phase 2 manual validation plan, Phase 3 role/language/experience policies, and source inventories.

### Action
Define deterministic group split rules. Primary isolation is by profile ID so the same candidate profile cannot appear in more than one split. Job family and language are stratification targets, not leakage-breaking keys. Manual validation labels must be sampled only after split assignment.

### Expected output
A split policy with ratios, seed, grouping key, isolation checks, leakage controls, and current profile-group inventory preview.

### Verification
The policy must make profile leakage impossible by construction and define checks that fail if any profile ID appears in multiple splits.


In [10]:
split_policy = {
    "schema_version": "pair-split-policy-v1",
    "seed": SPLIT_SEED,
    "primary_group_key": "profile_id",
    "secondary_stratification_keys": ["profile_role_family", "job_role_family", "language", "pair_type", "score_band"],
    "ratios": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "assignment_rule": "Assign each profile_id to exactly one split using a stable hash of seed + profile_id; all pairs for that profile stay in the same split.",
    "hash_buckets": {"train": [0, 69], "validation": [70, 84], "test": [85, 99]},
    "job_family_policy": "Track job_role_family balance inside each split. Optionally cap overrepresented job families, but never move a profile across splits to fix job balance.",
    "manual_validation_policy": "Sample manual-review rows from locked validation/test groups only after split assignment and artifact hashes are recorded.",
    "leakage_controls": [
        "No profile_id may appear in more than one split.",
        "fit_score and future manual labels cannot be used as split features.",
        "Company, application outcome, bookmark state, wrapper summaries, and hydrated job details are excluded from features and split assignment.",
        "Near-duplicate profiles must be reviewed before release if duplicate detection is added later.",
        "Evaluation reports must include pair_type, score_band, role_family, language, and experience slices per split.",
    ],
}


def split_name_for_profile(profile_id: Any) -> str:
    bucket = stable_bucket(profile_id)
    if bucket <= 69:
        return "train"
    if bucket <= 84:
        return "validation"
    return "test"

profile_split_preview = pd.DataFrame({
    "profile_id": profiles["ID"],
    "profile_role_family": profile_role_families,
    "experience": profiles["Experience"].fillna("UNKNOWN"),
})
profile_split_preview["split"] = profile_split_preview["profile_id"].map(split_name_for_profile)

split_group_inventory = {
    "profile_counts_by_split": {str(k): int(v) for k, v in profile_split_preview["split"].value_counts().items()},
    "profile_counts_by_split_and_role_family_top": {
        f"{split}:{family}": int(count)
        for (split, family), count in profile_split_preview.groupby(["split", "profile_role_family"]).size().sort_values(ascending=False).head(20).items()
    },
    "profile_counts_by_split_and_experience": {
        f"{split}:{experience}": int(count)
        for (split, experience), count in profile_split_preview.groupby(["split", "experience"]).size().items()
    },
}

split_acceptance = {
    "ratios_sum_to_one": round(sum(split_policy["ratios"].values()), 6) == 1.0,
    "primary_group_key_is_profile_id": split_policy["primary_group_key"] == "profile_id",
    "profile_ids_assigned_once": int(profile_split_preview["profile_id"].nunique()) == int(len(profile_split_preview)),
    "leakage_check_defined": any("No profile_id" in item for item in split_policy["leakage_controls"]),
}

assert all(split_acceptance.values()), split_acceptance
print(json.dumps({"split_acceptance": split_acceptance, "profile_counts_by_split": split_group_inventory["profile_counts_by_split"]}, indent=2))


{
  "split_acceptance": {
    "ratios_sum_to_one": true,
    "primary_group_key_is_profile_id": true,
    "profile_ids_assigned_once": true,
    "leakage_check_defined": true
  },
  "profile_counts_by_split": {
    "train": 48754,
    "test": 10727,
    "validation": 10448
  }
}


## Step 4.4 — Pair metadata

### Purpose
List required columns such as `pair_type`, label components, final score, language, role family, and split name.

### Required input
Use Phase 2 label components, Phase 3 normalization outputs, pair-type taxonomy, split policy, and feature-quality metrics.

### Action
Define a versioned pair metadata schema with identifiers for audit only, normalized fields, label component columns, quality flags, split fields, and artifact lineage. Mark which fields are allowed as model features and which are audit-only.

### Expected output
A required-column schema that pair-generation code must satisfy before baselines or model experiments run.

### Verification
The schema must include pair type, all job-fit label components, final score, language, role family, split name, and leakage/audit fields.


In [11]:
pair_metadata_schema = [
    {"column": "pair_id", "type": "string", "required": True, "use": "audit", "notes": "Stable deterministic pair identifier from dataset version + profile_id + job_id + pair_type."},
    {"column": "profile_id", "type": "string", "required": True, "use": "split_audit_only", "notes": "Primary group key; never a model feature."},
    {"column": "job_id", "type": "string", "required": True, "use": "audit_only", "notes": "Used for traceability and backend candidate validation; never a model feature."},
    {"column": "pair_type", "type": "category", "required": True, "use": "feature_or_slice", "notes": "One of the Phase 4 pair taxonomy values."},
    {"column": "split", "type": "category", "required": True, "use": "split_control", "notes": "train, validation, or test from profile group assignment."},
    {"column": "score_band", "type": "category", "required": True, "use": "slice", "notes": "low, medium, or high according to Phase 2 score-band policy."},
    {"column": "job_fit_score", "type": "float_0_1", "required": True, "use": "target", "notes": "Final weak-label v2 score for pair generation; API conversion to 0-100 happens later."},
    {"column": "skill_overlap_score", "type": "float_0_1", "required": True, "use": "label_component", "notes": "Phase 2 component."},
    {"column": "semantic_similarity_score", "type": "float_0_1", "required": True, "use": "label_component", "notes": "Phase 2 component when embeddings exist; otherwise explicit missing flag."},
    {"column": "experience_match_score", "type": "float_0_1", "required": True, "use": "label_component", "notes": "Phase 2 component using Phase 3 experience mapping."},
    {"column": "role_match_score", "type": "float_0_1", "required": True, "use": "label_component", "notes": "Phase 2 component using role-family policy."},
    {"column": "requirement_coverage_score", "type": "float_0_1", "required": True, "use": "label_component", "notes": "Phase 2 component."},
    {"column": "language", "type": "category", "required": True, "use": "slice", "notes": "ID, EN, MIXED, or UNKNOWN from Phase 3 policy."},
    {"column": "profile_role_family", "type": "category", "required": True, "use": "slice_or_feature", "notes": "Coarse candidate role family."},
    {"column": "job_role_family", "type": "category", "required": True, "use": "slice_or_feature", "notes": "Coarse job role family."},
    {"column": "profile_experience_band", "type": "category", "required": True, "use": "slice_or_feature", "notes": "Canonical experience band from Phase 3."},
    {"column": "job_experience_band", "type": "category", "required": True, "use": "slice_or_feature", "notes": "Canonical job seniority band from Phase 3."},
    {"column": "matched_skills", "type": "list[string]", "required": True, "use": "evidence_output", "notes": "Grounded matched skill evidence."},
    {"column": "missing_skills", "type": "list[string]", "required": True, "use": "evidence_output", "notes": "Grounded missing skill evidence."},
    {"column": "unknown_language", "type": "boolean", "required": True, "use": "quality_flag", "notes": "Feature-quality flag."},
    {"column": "unknown_experience", "type": "boolean", "required": True, "use": "quality_flag", "notes": "Feature-quality flag."},
    {"column": "empty_skills", "type": "boolean", "required": True, "use": "quality_flag", "notes": "Feature-quality flag."},
    {"column": "empty_text", "type": "boolean", "required": True, "use": "quality_flag", "notes": "Feature-quality flag."},
    {"column": "label_version", "type": "string", "required": True, "use": "lineage", "notes": "Example: jobfit_weak_label_v2."},
    {"column": "feature_config_version", "type": "string", "required": True, "use": "lineage", "notes": "Example: text-feature-construction-v1 + skill-normalization-v1."},
    {"column": "source_dataset_hash", "type": "string", "required": True, "use": "lineage", "notes": "Hash of source manifests/artifacts used to generate pairs."},
]

required_metadata_columns = {
    "pair_type", "skill_overlap_score", "semantic_similarity_score", "experience_match_score",
    "role_match_score", "requirement_coverage_score", "job_fit_score", "language",
    "profile_role_family", "job_role_family", "split",
}
metadata_columns = {row["column"] for row in pair_metadata_schema}
component_columns = {f"{component['component']}_score" for component in phase2_report["jobfit_label_components"]}
metadata_acceptance = {
    "required_columns_present": required_metadata_columns <= metadata_columns,
    "all_phase2_jobfit_components_present": component_columns <= metadata_columns,
    "audit_identifiers_marked_non_feature": all(
        row["use"] != "feature" for row in pair_metadata_schema if row["column"] in {"profile_id", "job_id"}
    ),
    "quality_flags_present": {"unknown_language", "unknown_experience", "empty_skills", "empty_text"} <= metadata_columns,
}

assert all(metadata_acceptance.values()), metadata_acceptance
print(json.dumps(metadata_acceptance, indent=2))


{
  "required_columns_present": true,
  "all_phase2_jobfit_components_present": true,
  "audit_identifiers_marked_non_feature": true,
  "quality_flags_present": true
}


## Step 4.5 — Distribution diagnostics

### Purpose
Plan charts for score distribution, score by pair type, score by role, score by language, and split balance.

### Required input
Use the target distribution, split policy, pair metadata schema, Phase 3 feature-quality metrics, and current source inventories.

### Action
Define required diagnostic charts/tables, source columns, release gates, and slice requirements. Diagnostics must be written before training and reviewed before Phase 5 baseline evaluation.

### Expected output
A diagnostics plan covering score distribution, score by pair type, score by role, score by language, split balance, high-fit coverage, and feature-quality flags.

### Verification
Diagnostics must include all acceptance-required views and define failure conditions for leakage, missing score bands, and unstable split balance.


In [12]:
distribution_diagnostics = [
    {"diagnostic": "score_distribution", "view": "histogram + table", "required_columns": ["job_fit_score", "score_band"], "gate": "All low/medium/high bands present in every split."},
    {"diagnostic": "score_by_pair_type", "view": "boxplot + count table", "required_columns": ["pair_type", "job_fit_score"], "gate": "Every required pair type has non-zero rows."},
    {"diagnostic": "score_by_role_family", "view": "heatmap/table", "required_columns": ["profile_role_family", "job_role_family", "job_fit_score"], "gate": "Top role families have validation/test coverage."},
    {"diagnostic": "score_by_language", "view": "boxplot + count table", "required_columns": ["language", "job_fit_score", "split"], "gate": "UNKNOWN language reported separately; ID/EN/MIXED not hidden."},
    {"diagnostic": "split_balance", "view": "stacked bars + table", "required_columns": ["split", "pair_type", "score_band", "profile_id"], "gate": "Profile IDs are isolated; pair-type and score-band shares do not drift beyond tolerance."},
    {"diagnostic": "high_fit_coverage", "view": "table", "required_columns": ["split", "job_fit_score", "pair_type"], "gate": "Validation and test splits each meet minimum high-fit example counts."},
    {"diagnostic": "feature_quality_flags", "view": "rate table", "required_columns": ["unknown_language", "unknown_experience", "empty_skills", "empty_text", "split"], "gate": "Block if unmapped experience exists; warn/block thresholds from Phase 3 are visible."},
    {"diagnostic": "leakage_audit", "view": "assertion report", "required_columns": ["profile_id", "job_id", "split"], "gate": "No profile_id appears in multiple splits; no target/manual label columns used in feature inputs."},
]

diagnostic_release_gates = {
    "profile_leakage": "fail if any profile_id appears in more than one split",
    "missing_pair_type": "fail if any required pair_type has zero rows overall or in validation/test unless documented blocker exists",
    "missing_score_band": "fail if low, medium, or high band is absent from validation/test",
    "high_fit_coverage": "fail if validation or test high-fit examples are below the minimum coverage policy",
    "unknown_experience": "fail if source values are unmapped by Phase 3 policy",
    "unknown_language": "warn at Phase 3 threshold; report UNKNOWN separately instead of folding into EN",
}

diagnostic_names = {row["diagnostic"] for row in distribution_diagnostics}
required_diagnostics = {
    "score_distribution", "score_by_pair_type", "score_by_role_family",
    "score_by_language", "split_balance",
}
diagnostics_acceptance = {
    "required_diagnostics_present": required_diagnostics <= diagnostic_names,
    "leakage_gate_defined": "profile_leakage" in diagnostic_release_gates,
    "feature_quality_included": "feature_quality_flags" in diagnostic_names,
    "high_fit_coverage_included": "high_fit_coverage" in diagnostic_names,
}

phase4_acceptance = {
    "pair_categories_cover_positive_medium_negative_cases": pair_type_acceptance["required_pair_types_present"] and pair_type_acceptance["bands_covered"],
    "split_rules_prevent_obvious_profile_leakage": split_acceptance["primary_group_key_is_profile_id"] and split_acceptance["leakage_check_defined"],
    "distribution_diagnostics_are_specified": diagnostics_acceptance["required_diagnostics_present"] and diagnostics_acceptance["leakage_gate_defined"],
}

assert all(diagnostics_acceptance.values()), diagnostics_acceptance
assert all(phase4_acceptance.values()), phase4_acceptance

phase4_report = {
    "schema_version": "phase-04-pair-generation-splits-v1",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_snapshot": "legacy",
    "inputs": {
        "phase2_report": str(PHASE2_REPORT.relative_to(ROOT)),
        "phase3_report": str(PHASE3_REPORT.relative_to(ROOT)),
        "jobs_csv": str(JOBS_CSV.relative_to(ROOT)),
        "profiles_csv": str(PROFILES_CSV.relative_to(ROOT)),
        "todo_scope": "Phase 4 — Pair Generation and Split Strategy",
    },
    "source_inventory": source_inventory,
    "pair_type_taxonomy": pair_type_taxonomy,
    "target_score_distribution": score_distribution_counts,
    "pair_type_target_distribution": pair_type_distribution_counts,
    "minimum_high_fit_coverage": minimum_high_fit_coverage,
    "split_policy": split_policy,
    "split_group_inventory": split_group_inventory,
    "pair_metadata_schema": pair_metadata_schema,
    "distribution_diagnostics": distribution_diagnostics,
    "diagnostic_release_gates": diagnostic_release_gates,
    "blocked_until_later_phases": [
        "Pair rows are not generated in this planning notebook; implementation code must satisfy this schema before Phase 5 baselines.",
        "Manual validation labels must be sampled after group-safe split assignment and must not leak into training features.",
        "Legacy weak-label fit_score remains prototype-only evidence and must not define high-fit pair selection.",
        "Phase 5 must compare baselines across pair_type, role_family, language, experience, and score_band slices before model training.",
    ],
    "acceptance": phase4_acceptance,
}

write_json(PHASE4_REPORT, phase4_report)
print(f"Wrote {PHASE4_REPORT.relative_to(ROOT)}")
print(json.dumps(phase4_acceptance, indent=2))


Wrote reports/phase_04_pair_generation_splits.json
{
  "pair_categories_cover_positive_medium_negative_cases": true,
  "split_rules_prevent_obvious_profile_leakage": true,
  "distribution_diagnostics_are_specified": true
}


## Acceptance criteria

- [x] Pair categories cover positive, medium, and negative cases.
- [x] Split rules prevent obvious profile leakage.
- [x] Distribution diagnostics are specified.


## Phase notes

Phase 4 completes the pair-generation and split strategy. It intentionally does not create pair rows or train a model.

Follow-up work:

- Pair-generation implementation must use this taxonomy and metadata schema before Phase 5 baselines run.
- High-fit coverage is mandatory because the legacy pair dataset has no labels above the high-fit threshold.
- Split assignment must remain profile-grouped even when role-family or language balancing is imperfect.
- Manual validation candidates must be sampled after split assignment and artifact hashes are recorded.
